In [12]:
from openai import OpenAI
import requests
from minsearch import Index
import json

### Data pipeline

In [2]:
# data preprocessing
docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [3]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [4]:
index = Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [5]:
openai_client = OpenAI()

In [6]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [7]:
def search(query):
    """
    Retrieves 5 most relevant documents given a user query
    """
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )

    return results

In [13]:
instructions = """
    You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
    Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

prompt_template = """
    <QUESTION>
    {question}
    </QUESTION>

    <CONTEXT>
    {context}
    </CONTEXT>
""".strip()

def build_prompt(question, search_results):
    """
    Combines user query and search engine results into a formatted LLM prompt
    """
    search_json = json.dumps(search_results)
    return prompt_template.format(
        question=question,
        context=search_json
    )

In [9]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions=instructions)
    return answer

In [10]:
question = "how do I install Kafka in Python?"

In [14]:
rag(question)

'To install Kafka in Python, you can use the following command:\n\n- For the Confluent Kafka Python client, run:\n  ```bash\n  pip install confluent-kafka\n  ```\n\nAlternatively, if you prefer using Anaconda, execute:\n```bash\nconda install conda-forge::python-confluent-kafka\n```\n\nFor general Kafka functionality, if you face compatibility issues with the `kafka-python` package, you can install the package directly from the GitHub repository using:\n```bash\npip install git+https://github.com/dpkp/kafka-python.git\n```\nMake sure to uninstall any existing version of `kafka-python` first with:\n```bash\npip uninstall kafka-python\n```'